## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of Reinforcement Learning fundamentals, including MDPs, value functions, and policy optimization.
* Apply advanced Deep Reinforcement Learning algorithms (e.g., PPO) to solve complex control problems.
* Implement and critically evaluate components of the Reinforcement Learning from Human Feedback (RLHF) pipeline for aligning large language models.


## Final Assessment: Mastering Reinforcement Learning

Welcome to the final assessment for FT-03: Mastering Reinforcement Learning. This assessment is designed to evaluate your comprehensive understanding of the course material, from foundational concepts to advanced topics like Proximal Policy Optimization (PPO) and Reinforcement Learning from Human Feedback (RLHF).

As ML engineers working on model alignment and advanced training, your ability to grasp these complex methodologies and apply them practically is paramount. This assessment will challenge you to:

1.  **Recall and explain core RL concepts:** Demonstrate your theoretical knowledge of Markov Decision Processes, value functions, policy gradients, and the exploration-exploitation dilemma.
2.  **Analyze and compare advanced algorithms:** Articulate the nuances of algorithms like PPO, understanding their advantages, limitations, and practical considerations.
3.  **Design and implement RLHF components:** Apply your knowledge to a practical, albeit simplified, scenario involving the alignment of a language model using human feedback principles.

This assessment is structured into two main parts:

*   **Review Questions:** A series of conceptual questions to test your theoretical understanding across the course topics.
*   **Capstone Coding Problem:** A practical, hands-on coding challenge that requires you to integrate multiple concepts learned throughout the course, focusing on the RLHF pipeline.

Your solutions should reflect a deep understanding, clarity of thought, and adherence to best practices in both theoretical explanation and code implementation. Good luck!


### Part 1: Review Questions

Answer the following questions concisely and comprehensively. Aim for clarity and precision in your explanations.

1.  **On-Policy vs. Off-Policy RL:** What is the fundamental distinction between on-policy and off-policy Reinforcement Learning algorithms? Provide one example of each type and briefly explain why it falls into that category.

2.  **Exploration-Exploitation Dilemma:** Describe the exploration-exploitation dilemma in the context of RL. How do common strategies like epsilon-greedy, UCB (Upper Confidence Bound), or entropy regularization attempt to balance these two competing objectives?

3.  **Proximal Policy Optimization (PPO):** Explain the core idea behind Proximal Policy Optimization (PPO). What is the purpose of the clipping mechanism in PPO's objective function, and how does it address issues present in earlier policy gradient methods like vanilla Policy Gradients or A2C?

4.  **Reinforcement Learning from Human Feedback (RLHF) Pipeline:** Outline the typical stages involved in a Reinforcement Learning from Human Feedback (RLHF) pipeline for aligning a Large Language Model (LLM). For each stage, briefly describe its purpose and key outputs.

5.  **Reward Model in RLHF:** What is a "reward model" in the context of RLHF, and why is it a crucial component? How is a reward model typically trained, and what kind of data is required for its training?

6.  **Value Function Approximation:** Discuss the necessity of value function approximation in Deep Reinforcement Learning. How does using neural networks for value function approximation enable RL to scale to high-dimensional state spaces, and what challenges does it introduce?

7.  **Actor-Critic Methods:** Explain the roles of the "actor" and the "critic" in actor-critic methods. How do these two components interact to facilitate policy learning, and what are the benefits of this architecture compared to pure policy-gradient or pure value-based methods?

8.  **Generalization in RL:** In the context of complex, real-world environments, how do techniques like state representation learning, auxiliary tasks, or transfer learning contribute to better generalization in RL agents? Provide a brief example for one of these techniques.


### Part 2: Capstone Coding Problem - Simplified RLHF for Text Generation

**Scenario:**

As an ML engineer, you're tasked with aligning a small, pre-trained language model (LLM) to generate responses that are considered 'helpful' and 'harmless' based on human preferences. For this assessment, we will simulate a simplified version of the RLHF pipeline. Your goal is to implement the core logic of a PPO-based fine-tuning process, where the 'human feedback' is provided by a simulated reward model.

**Objective:**

Implement a simplified RLHF training loop. You will need to:

1.  **Simulate a Language Model Policy:** Create a simple `torch.nn.Module` that acts as our LLM policy. This policy will take a dummy input (representing a prompt or state) and output a probability distribution over a small vocabulary of tokens. It should be able to generate a short sequence of tokens.
2.  **Implement a Simulated Reward Model:** Design a `RewardModel` class (or function) that takes a generated sequence of tokens and returns a scalar reward. This reward model will simulate human preferences by assigning higher rewards to 'good' sequences and lower rewards to 'bad' sequences based on simple heuristics (e.g., presence of specific tokens, sequence length, etc.).
3.  **Develop a PPO Agent:** Create a `PPOAgent` class that can interact with the simulated LLM policy and the reward model. This agent should be capable of:
    *   Collecting trajectories (generated sequences, their log probabilities, and rewards).
    *   Performing a PPO update on the LLM policy's parameters using the collected data and the reward model's feedback.
4.  **Simulate the RLHF Training Loop:** Orchestrate the interaction between the PPO agent, the LLM policy, and the reward model over several training iterations to demonstrate the learning process.

**Simplifications for this Assessment:**

*   You do **not** need to load or fine-tune an actual large language model. Your 'LLM policy' can be a small neural network that outputs token probabilities.
*   The 'state' or 'prompt' can be a fixed dummy tensor.
*   The 'vocabulary' can be a small set of integers.
*   The reward model will be heuristic-based, not a trained neural network.
*   You can simplify the Advantage calculation (e.g., use raw rewards or a simple baseline).

**Expected Output:**

Your solution should include the `RewardModel` and `PPOAgent` classes, along with a main training loop that demonstrates the policy learning to generate higher-reward sequences over time. Print out the average reward per epoch to show progress.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributions as distributions
import numpy as np

# Define hyperparameters
VOCAB_SIZE = 10
SEQUENCE_LENGTH = 5
EMBEDDING_DIM = 16
HIDDEN_DIM = 32

# --- Simulated LLM Policy --- #
class LLMPolicy(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        # This is a highly simplified 'LLM' that takes a dummy input
        # and outputs logits for the next token.
        # For simplicity, we'll assume a fixed 'state' or 'prompt' embedding.
        self.embedding = nn.Embedding(1, embedding_dim) # Dummy embedding for a single 'start' token
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_embedding, hidden=None):
        # input_embedding: (batch_size, 1, embedding_dim)
        output, hidden = self.rnn(input_embedding, hidden)
        logits = self.output_layer(output.squeeze(1)) # (batch_size, vocab_size)
        return logits, hidden

    def generate_sequence(self, batch_size, sequence_length):
        # Generate a sequence of tokens
        sequences = []
        log_probs_list = []
        hidden = None
        # Start with a dummy input (e.g., a fixed start token embedding)
        current_input_embedding = self.embedding(torch.zeros(batch_size, dtype=torch.long)).unsqueeze(1)

        for _ in range(sequence_length):
            logits, hidden = self.forward(current_input_embedding, hidden)
            dist = distributions.Categorical(logits=logits)
            action = dist.sample() # Sample a token
            log_prob = dist.log_prob(action)

            sequences.append(action.unsqueeze(1))
            log_probs_list.append(log_prob.unsqueeze(1))

            # For the next step, we'd typically embed the sampled action.
            # For this simplified model, let's just use a fixed input for simplicity
            # or you can embed the sampled token if you want more complexity.
            # current_input_embedding = self.embedding(action).unsqueeze(1) # More realistic
            current_input_embedding = self.embedding(torch.zeros(batch_size, dtype=torch.long)).unsqueeze(1) # Simplified

        return torch.cat(sequences, dim=1), torch.cat(log_probs_list, dim=1)

# --- Simulated Reward Model --- #
class RewardModel:
    def __init__(self, good_tokens, bad_tokens):
        self.good_tokens = set(good_tokens)
        self.bad_tokens = set(bad_tokens)

    def get_reward(self, sequence):
        # sequence: a 1D tensor of tokens for a single generated sequence
        reward = 0.0
        for token in sequence:
            if token.item() in self.good_tokens:
                reward += 1.0
            elif token.item() in self.bad_tokens:
                reward -= 1.5 # Penalize bad tokens more heavily
        
        # Add a small bonus for longer sequences (if they're not all bad)
        if len(sequence) > 0 and reward > -len(sequence) * 1.5: # Only if not entirely penalized
            reward += len(sequence) * 0.1
            
        return torch.tensor(reward, dtype=torch.float32)

# --- PPO Agent --- #
class PPOAgent:
    def __init__(self, policy_model, optimizer, clip_epsilon, gamma, k_epochs):
        self.policy = policy_model
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.gamma = gamma # Discount factor
        self.k_epochs = k_epochs # Number of PPO update epochs

        # Placeholder for old policy (for ratio calculation)
        self.old_policy = type(policy_model)(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
        self.old_policy.load_state_dict(self.policy.state_dict())

    def collect_trajectories(self, batch_size, sequence_length):
        # Implement trajectory collection: generate sequences, get rewards, store log_probs
        # This method should return collected sequences, their log_probabilities, and rewards.
        pass # YOUR IMPLEMENTATION HERE

    def update_policy(self, sequences, old_log_probs, rewards):
        # Implement the PPO update logic
        # Calculate advantages, compute PPO loss, perform optimization step
        pass # YOUR IMPLEMENTATION HERE

# --- Training Loop --- #
if __name__ == "__main__":
    # Hyperparameters for training
    LEARNING_RATE = 1e-3
    PPO_CLIP_EPSILON = 0.2
    PPO_GAMMA = 0.99
    PPO_K_EPOCHS = 4
    BATCH_SIZE = 64
    NUM_EPISODES = 100

    # Initialize components
    policy_model = LLMPolicy(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
    optimizer = optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
    
    # Define some 'good' and 'bad' tokens for the reward model
    # E.g., tokens 0, 1 are good; tokens 8, 9 are bad.
    reward_model = RewardModel(good_tokens=[0, 1], bad_tokens=[8, 9])

    ppo_agent = PPOAgent(policy_model, optimizer, PPO_CLIP_EPSILON, PPO_GAMMA, PPO_K_EPOCHS)

    print("Starting RLHF training simulation...")
    for episode in range(NUM_EPISODES):
        # 1. Collect trajectories
        sequences, old_log_probs, rewards = ppo_agent.collect_trajectories(BATCH_SIZE, SEQUENCE_LENGTH)

        # 2. Update the policy using PPO
        ppo_agent.update_policy(sequences, old_log_probs, rewards)

        if (episode + 1) % 10 == 0:
            avg_reward = rewards.mean().item()
            print(f"Episode {episode + 1}/{NUM_EPISODES}, Average Reward: {avg_reward:.2f}")

    print("Training simulation complete.")

    # Optional: Generate a final sequence to see if it improved
    print("\nFinal generated sequence example:")
    with torch.no_grad():
        final_sequence, _ = policy_model.generate_sequence(1, SEQUENCE_LENGTH)
        print(f"Sequence: {final_sequence.squeeze().tolist()}")
        print(f"Reward: {reward_model.get_reward(final_sequence.squeeze()).item():.2f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributions as distributions
import numpy as np

# Define hyperparameters
VOCAB_SIZE = 10
SEQUENCE_LENGTH = 5
EMBEDDING_DIM = 16
HIDDEN_DIM = 32

# --- Simulated LLM Policy --- #
class LLMPolicy(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        # This is a highly simplified 'LLM' that takes a dummy input
        # and outputs logits for the next token.
        # For simplicity, we'll assume a fixed 'state' or 'prompt' embedding.
        self.embedding = nn.Embedding(1, embedding_dim) # Dummy embedding for a single 'start' token
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_embedding, hidden=None):
        # input_embedding: (batch_size, 1, embedding_dim)
        output, hidden = self.rnn(input_embedding, hidden)
        logits = self.output_layer(output.squeeze(1)) # (batch_size, vocab_size)
        return logits, hidden

    def generate_sequence(self, batch_size, sequence_length):
        # Generate a sequence of tokens
        sequences = []
        log_probs_list = []
        hidden = None
        # Start with a dummy input (e.g., a fixed start token embedding)
        # We'll use a fixed 'start token' ID 0 for simplicity
        current_input_embedding = self.embedding(torch.zeros(batch_size, dtype=torch.long)).unsqueeze(1)

        for _ in range(sequence_length):
            logits, hidden = self.forward(current_input_embedding, hidden)
            dist = distributions.Categorical(logits=logits)
            action = dist.sample() # Sample a token
            log_prob = dist.log_prob(action)

            sequences.append(action.unsqueeze(1))
            log_probs_list.append(log_prob.unsqueeze(1))

            # For the next step, embed the sampled action to feed back into the RNN
            current_input_embedding = self.embedding(action).unsqueeze(1)

        return torch.cat(sequences, dim=1), torch.cat(log_probs_list, dim=1)

# --- Simulated Reward Model --- #
class RewardModel:
    def __init__(self, good_tokens, bad_tokens):
        self.good_tokens = set(good_tokens)
        self.bad_tokens = set(bad_tokens)

    def get_reward(self, sequence):
        # sequence: a 1D tensor of tokens for a single generated sequence
        reward = 0.0
        for token in sequence:
            if token.item() in self.good_tokens:
                reward += 1.0
            elif token.item() in self.bad_tokens:
                reward -= 1.5 # Penalize bad tokens more heavily
        
        # Add a small bonus for longer sequences (if they're not all bad)
        # This encourages the model to generate full sequences, not just stop early if possible
        if len(sequence) > 0 and reward > -len(sequence) * 1.5: # Only if not entirely penalized
            reward += len(sequence) * 0.1
            
        return torch.tensor(reward, dtype=torch.float32)

# --- PPO Agent --- #
class PPOAgent:
    def __init__(self, policy_model, optimizer, clip_epsilon, gamma, k_epochs):
        self.policy = policy_model
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.gamma = gamma # Discount factor
        self.k_epochs = k_epochs # Number of PPO update epochs

        # Initialize an 'old_policy' to store the policy parameters before an update
        # This is crucial for calculating the ratio in PPO.
        self.old_policy = type(policy_model)(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
        self.old_policy.load_state_dict(self.policy.state_dict())

    def collect_trajectories(self, batch_size, sequence_length, reward_model):
        # Set policy to evaluation mode for data collection
        self.policy.eval()
        
        # Generate sequences and their log probabilities using the current policy
        with torch.no_grad():
            sequences, log_probs = self.policy.generate_sequence(batch_size, sequence_length)
        
        # Calculate rewards for each generated sequence using the reward model
        rewards = torch.tensor([reward_model.get_reward(seq) for seq in sequences], dtype=torch.float32)
        
        # Set policy back to training mode
        self.policy.train()
        
        return sequences, log_probs, rewards

    def update_policy(self, sequences, old_log_probs, rewards):
        # Update the old policy's state dict to match the current policy before starting PPO epochs
        self.old_policy.load_state_dict(self.policy.state_dict())
        
        # Normalize rewards for better stability (optional but common)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-5)

        # PPO update loop for k_epochs
        for _ in range(self.k_epochs):
            # Calculate current log probabilities for the collected sequences
            # We need to re-run generation to get current log_probs, but for simplicity
            # and to avoid re-sampling, we'll re-evaluate the log_probs of the *already sampled* actions.
            # This requires re-running the forward pass for each token in the sequence.
            
            current_log_probs_list = []
            hidden = None
            current_input_embedding = self.policy.embedding(torch.zeros(sequences.size(0), dtype=torch.long)).unsqueeze(1)
            
            for t in range(SEQUENCE_LENGTH):
                logits, hidden = self.policy(current_input_embedding, hidden)
                dist = distributions.Categorical(logits=logits)
                # Get log_prob for the *sampled* action (token) from the current policy
                current_log_probs_list.append(dist.log_prob(sequences[:, t]).unsqueeze(1))
                current_input_embedding = self.policy.embedding(sequences[:, t]).unsqueeze(1)
            
            current_log_probs = torch.cat(current_log_probs_list, dim=1)
            
            # Sum log probabilities over the sequence to get sequence-level log_prob
            # This is a simplification; in full RLHF, it's often token-level or sequence-level with attention.
            current_log_probs_sum = current_log_probs.sum(dim=1)
            old_log_probs_sum = old_log_probs.sum(dim=1)

            # Calculate the ratio r(theta) = pi_theta(a|s) / pi_theta_old(a|s)
            # We use .exp() because we are working with log probabilities
            ratios = torch.exp(current_log_probs_sum - old_log_probs_sum.detach())

            # Calculate the clipped surrogate objective
            # Advantages are simply the rewards in this simplified setup
            advantages = rewards # In a full PPO, this would be GAE or similar

            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages
            
            # PPO loss: take the minimum of the two surrogates, and we want to maximize it,
            # so we negate it for gradient descent (minimization).
            policy_loss = -torch.min(surr1, surr2).mean()

            # Perform optimization step
            self.optimizer.zero_grad()
            policy_loss.backward()
            self.optimizer.step()

# --- Training Loop --- #
if __name__ == "__main__":
    # Hyperparameters for training
    LEARNING_RATE = 1e-3
    PPO_CLIP_EPSILON = 0.2
    PPO_GAMMA = 0.99 # Discount factor (not heavily used in this simplified setup, but good practice)
    PPO_K_EPOCHS = 4 # Number of PPO optimization steps per data collection
    BATCH_SIZE = 64
    NUM_EPISODES = 200 # More episodes to see clearer learning

    # Initialize components
    policy_model = LLMPolicy(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
    optimizer = optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
    
    # Define some 'good' and 'bad' tokens for the reward model
    # E.g., tokens 0, 1, 2 are good; tokens 7, 8, 9 are bad.
    reward_model = RewardModel(good_tokens=[0, 1, 2], bad_tokens=[7, 8, 9])

    ppo_agent = PPOAgent(policy_model, optimizer, PPO_CLIP_EPSILON, PPO_GAMMA, PPO_K_EPOCHS)

    print("Starting RLHF training simulation...")
    print(f"Initial policy parameters (first few): {policy_model.output_layer.weight.data.flatten()[:5].tolist()}")

    for episode in range(NUM_EPISODES):
        # 1. Collect trajectories using the current policy
        sequences, old_log_probs, rewards = ppo_agent.collect_trajectories(BATCH_SIZE, SEQUENCE_LENGTH, reward_model)

        # 2. Update the policy using PPO with the collected data
        ppo_agent.update_policy(sequences, old_log_probs, rewards)

        if (episode + 1) % 20 == 0:
            avg_reward = rewards.mean().item()
            print(f"Episode {episode + 1}/{NUM_EPISODES}, Average Reward: {avg_reward:.2f}")

    print("Training simulation complete.")
    print(f"Final policy parameters (first few): {policy_model.output_layer.weight.data.flatten()[:5].tolist()}")

    # Optional: Generate a final sequence to see if it improved
    print("\nFinal generated sequence example:")
    with torch.no_grad():
        final_sequence, _ = policy_model.generate_sequence(1, SEQUENCE_LENGTH)
        print(f"Sequence: {final_sequence.squeeze().tolist()}")
        print(f"Reward: {reward_model.get_reward(final_sequence.squeeze()).item():.2f}")

    print("\nAnother example:")
    with torch.no_grad():
        final_sequence_2, _ = policy_model.generate_sequence(1, SEQUENCE_LENGTH)
        print(f"Sequence: {final_sequence_2.squeeze().tolist()}")
        print(f"Reward: {reward_model.get_reward(final_sequence_2.squeeze()).item():.2f}")
